# Assignment 1: Planetary Motion with `Vec3D`

**6EMA02 Particle-based Simulations, 2026-2027 edition**
N-body gravity in C · integrators · conservation · a real asteroid

## How this assignment is assessed: read this first

This notebook is both the assignment text and your report. Task cells
(like this one) are the assignment text and are marked read-only. Jupyter
enforces that; some editors (VS Code among them) do not, so simply leave
them untouched either way: hand-ins are checked against the released
notebook, and modified task cells are flagged. You write your answers in
the cells below each task and hand in this same notebook, **executed**,
together with an HTML export, your code, and the data files the notebook
loads (layout at the end of this notebook).

Your report is **not graded on its content**. Its result is determined in
the individual oral examination at the end of the quartile: the exam is a
conversation about *your* code and *your* results, with this notebook on
screen. Write it as your defence dossier: concise, bullet answers are
fine, every claim something you can stand behind. A report you never hand
in scores 0 for its weight in the final grade; a handed-in report's
result *is* your oral grade.

**AI and sources.** You may use AI assistants (ChatGPT, Claude, Copilot,
…) freely in this course, for code, analysis, and text. Declare the use
briefly in the *Sources* section below, along with any other external
sources: anything in your submission (code, text, data, results) that
your group did not produce itself. Declared sources are never penalised:
nobody grades the text of your report or the style of your code. The oral
examination assesses what *you* understand about the simulation you hand
in. Anything in your submission is fair game, and "we didn't write that
part" is not an available answer. Presenting undeclared external work as
your own is academic misconduct and is treated as such.

## Group

- **Group ID:** …
- **Members:** name (student ID), name (student ID)

## Setting up your working environment

You need two toolchains: a C compiler for the simulation, and Python with
Jupyter for the analysis in this notebook.

**C compiler.** Linux and macOS have `gcc` (or `clang` aliased to it).
On Windows, install MSYS2 and use its `gcc`, or work inside WSL. Check
with `gcc --version`.

**Python.** Work in a virtual environment so this course cannot break
your other Python installations. From the directory holding this
notebook, with `requirements.txt` (Files ▸ Assignments) next to it:

```
python3 -m venv pbs-env
source pbs-env/bin/activate          # Windows: pbs-env\Scripts\activate
pip install -r requirements.txt
jupyter lab
```

That installs NumPy, SciPy, matplotlib, pandas, Jupyter, and the two
helper packages used below. Every session afterwards starts with the
`activate` line. In Jupyter or VS Code, make sure the notebook's kernel
is the `pbs-env` one before you run anything; a notebook running on some
other interpreter is the most common cause of "it works for me but not
for you".

**Two helper scripts** (Files ▸ Assignments), both run from the
directory holding this notebook:

- `nb2html.py` produces the HTML you hand in:
  `python3 nb2html.py A1.ipynb`. It embeds your figures and writes the
  formulas so that they display in Canvas, which blocks the script that
  a plain `jupyter nbconvert` export relies on.
- `check_my_submission.py` checks your zip against the hand-in rules
  *before* you submit: `python3 check_my_submission.py group7_A1.zip
  --release A1.ipynb`, where `--release` points at a freshly downloaded,
  unmodified copy of this notebook. Run it; it catches the mistakes that
  are annoying to repair afterwards.

## The physics: what you are actually solving

Two bodies with masses $m_i$ and $m_j$ at positions $\mathbf{r}_i$ and
$\mathbf{r}_j$ attract each other along the line connecting them.
**Newton's law of universal gravitation** gives the force on body $i$
exerted by body $j$:

$$\mathbf{F}_{ij} = G\, \frac{m_i m_j}{r_{ij}^{3}}\,
\left(\mathbf{r}_j - \mathbf{r}_i\right), \qquad
r_{ij} = \lVert \mathbf{r}_j - \mathbf{r}_i \rVert ,$$

with the gravitational constant $G$. The force points from $i$ towards
$j$ and falls off as $1/r_{ij}^2$ in magnitude. By Newton's third law
$\mathbf{F}_{ji} = -\mathbf{F}_{ij}$, which is why one evaluation per
*pair* suffices. The associated potential energy of the pair is
$-G m_i m_j / r_{ij}$.

With $N$ bodies, each one feels the sum of the pulls of all others, and
**Newton's second law** turns this into a system of coupled second-order
differential equations, the equations of motion:

$$m_i \frac{\mathrm{d}^2 \mathbf{r}_i}{\mathrm{d}t^2} =
\sum_{j \neq i} \mathbf{F}_{ij}
\qquad \Longleftrightarrow \qquad
\frac{\mathrm{d}^2 \mathbf{r}_i}{\mathrm{d}t^2} =
\mathbf{a}_i = \sum_{j \neq i} G\, \frac{m_j}{r_{ij}^{3}}\,
\left(\mathbf{r}_j - \mathbf{r}_i\right).$$

Note what happened in the second form: the mass of body $i$ cancelled.
Its acceleration depends only on the masses of the *other* bodies. Keep
that form in your code (task A2 returns to it).

For $N = 2$ this system is solvable in closed form and gives Kepler's
ellipses. For $N \geq 3$ it is not: there is no general analytical
solution, and the only way forward is to **integrate the equations
numerically**, stepping the state forward in small time steps
$\Delta t$. That is the entire subject of this course. Gravity is the
gentlest possible starting point, because the force law is simple, but
the program you write here has the same skeleton as the molecular
dynamics code of A2: a pair loop that computes forces, an integrator
that advances positions and velocities, and diagnostics that tell you
whether to believe the result.

Because the equations are second order in time, you need **two** vectors
per body to start: a position and a velocity. Together, for all bodies,
these form the *initial state*, and the whole simulation is an initial
value problem. Where that initial state comes from is the subject of the
next cell.

## The data: JPL Horizons snapshots

The initial states are real measurements of the real solar system,
provided as snapshot files.

**Horizons** is the online ephemeris service of NASA's Jet Propulsion
Laboratory (<https://ssd.jpl.nasa.gov/horizons/>). An *ephemeris* is a
table of the positions and velocities of solar-system bodies as a
function of time. Behind Horizons sits JPL's DE441 planetary ephemeris:
a numerical integration of the solar system, fitted to decades of radar
ranging, spacecraft tracking and optical astrometry. For the planets it
is accurate to well under a kilometre over our time span, so for this
assignment Horizons is *the truth*: it is the external reference against
which you **validate** your simulation, exactly the distinction the
lecture notes draw between verification (is my code solving the
equations right?) and validation (are these the right equations, with
the right data?).

Two snapshot files are provided (Files ▸ Data), generated with the
provided `get_from_Horizons.py`:

- `bodies_2026-09-01.dat`: the starting state, 12 bodies, including the
  asteroid 99942 Apophis with mass 0.
- `bodies_2027-09-01.dat`: the same system exactly one year later, for
  the comparison in task C2.

Each line is one body: `id name mass_kg x y z vx vy vz`, in SI units
(metres, metres per second, kilograms), in the ecliptic J2000 frame with
the solar-system barycentre as origin. Lines starting with `#` are a
header describing exactly this. Times are TDB (a uniform dynamical time
scale), which for our purposes differs from UTC by about a minute.

Three details of the data that matter for your results:

1. **Use $G = 6.6743015 \times 10^{-11}\ \mathrm{m^3\,kg^{-1}\,s^{-2}}$.**
   The masses were derived from the $GM$ values Horizons publishes,
   using this $G$, so with it your products $G m$ reproduce those $GM$
   values exactly. Astronomy measures $GM$ far more precisely than $G$
   and $m$ separately.
2. For the moon-bearing outer planets (Jupiter through Pluto) the files
   give the planetary-*system* barycentre and the total system mass
   (planet plus moons), not the planet centre. Think about why that is
   the right point to propagate in a model that contains no moons except
   our own; task C2 asks you about it.
3. **99942 Apophis** is a roughly 370 m asteroid discovered in 2004. On
   13 April 2029 it passes Earth closer than the geostationary satellite
   ring, one of the closest approaches by an object this size in recorded
   history. Its mass is negligible compared with any planet, so the file
   lists it as 0: it is a test particle, which needs no special-casing
   provided you write the equations of motion as above.

## What you will build, and the required vector type

You will write a C program that integrates the equations of motion for
the bodies in a snapshot file, using *velocity-Verlet* and, for
comparison, *Euler-forward*; monitor the conserved quantities; validate
against Horizons; and finally predict the 2029 Apophis close approach.

**Requirement:** all positions, velocities, accelerations, and forces use

```c
typedef struct Vec3D { double x, y, z; } Vec3D;
```

with dynamically allocated arrays (`Vec3D *r, *v, *a;`) and small inline
helper functions for the vector operations:

```c
static inline Vec3D v3(double x, double y, double z);
static inline Vec3D add(Vec3D a, Vec3D b);
static inline Vec3D sub(Vec3D a, Vec3D b);
static inline Vec3D scl(double s, Vec3D a);
static inline double dot(Vec3D a, Vec3D b);
static inline Vec3D cross(Vec3D a, Vec3D b);
static inline double norm(Vec3D a);
```

Writing vector code this way, rather than juggling `x`, `y`, `z`
separately, is what keeps the force routine short enough to read and to
debug. The same type reappears in every code of this course.

## Code map: fill in as you go

The examiner uses this table in the oral to jump straight to your code,
so **make the entries clickable links**: a markdown link with a path
relative to this notebook, for example

```
| B4 forces | [`compute_acc_and_potential`](code/nbody.c) | [run.csv](data/run.csv) |
```

Relative paths only (`code/…`, `data/…`). A link like
`/home/you/pbs/nbody.c` or `C:\Users\you\...` works on your machine and
nowhere else, and everything you link to must be inside the zip you hand
in. `check_my_submission.py` verifies this for you.

| Task | File / function(s) | Data files |
|------|--------------------|------------|
| B3 input reader | … | … |
| B4 forces + potential | … | … |
| B5 integrators | … | … |
| B6 diagnostics | … | … |
| C2 one-year comparison | … | … |
| C5 Apophis | … | … |

## Sources

**AI tools used:** … / none.
**Used for:** … .
**Other external sources** (code, text, data, or results not produced by
us): … / none.
**Parts of the submission with substantial external content:** … .
**What we did ourselves to verify these parts:** … .

## Build & run log

Exact compile and run commands, machine, and approximate wall-clock time
for each production run whose output sits in `data/`:

- …

## Part A: Modeling (short written answers)

*In the oral you should be able to:* explain why each conserved quantity
is conserved, and which kinds of bug each one can and cannot catch.

### A1) Force algorithm

Write pseudocode, or extensively commented C code, for computing the
gravitational accelerations and the total potential energy with `Vec3D`.

Address explicitly:

- where Newton's third law enters your loop, and which two lines of your
  code implement it;
- the number of pair evaluations per step as a formula in $N$, evaluated
  for the 12 bodies of the snapshot;
- how the potential energy is accumulated in the same loop.

*Your answer (A1):* …

### A2) Zero-mass bodies

Explain how to treat entries with $m = 0$: they feel gravity but exert
none. Why can you not compute their acceleration as $\mathbf{F}/m$, and
what does your code compute instead? State what a program that divides
by the mass would produce for Apophis, and how you would recognise that
value in your output.

*Your answer (A2):* …

### A3) Integrator formulas

Give the update rules for velocity-Verlet and Euler-forward in terms of
vector operations, and state for each how many force evaluations one
step costs. Mark in the velocity-Verlet scheme where exactly the force
evaluation sits, and say what goes wrong if it sits elsewhere.

*Your answer (A3):* …

### A4) Invariants

Give formulas for $K$, $U$, $E = K + U$, the total angular momentum
$\mathbf{L}$, and $\mathbf{R}_\mathrm{COM}$.

Then answer **as a table**, one row per invariant, with columns: exact
dynamics, velocity-Verlet, Euler-forward, and *what kind of
implementation error a drift in this quantity would signal*. Note
carefully which invariants are conserved *exactly* (up to round-off) by
velocity-Verlet and which only approximately, and which of them notice
the size of the time step at all.

*Your answer (A4):* …

## Part B: C implementation

*In the oral you should be able to:* walk through any function you wrote;
explain your memory layout and what must happen after every `malloc`;
justify where the force evaluation sits inside the velocity-Verlet step.

### B1) Data structures

Dynamically allocate the arrays; keep only the current positions,
velocities, and accelerations in memory (no trajectory history).

### B2) Vector helpers

Implement all helpers listed above and use them consistently.

### B3) Input

Implement `read_initial_conditions(...)` for the snapshot files. Lines
starting with `#` are comments (the header states the columns and
units); skip them.

### B4) Forces

Implement `compute_acc_and_potential(...)` returning accelerations and
the total potential energy, with correct zero-mass handling.

### B5) Integrator switch

An enum plus runtime flag selects Euler-forward or velocity-Verlet.

### B6) Diagnostics

Compute $K$, $U$, $E$, $\mathbf{L}$, $\mathbf{R}_\mathrm{COM}$ per step;
write to CSV.

### B7) Trajectory output

Write positions to file every fixed number of steps (user-selectable).

Document the result in the cell below: list the files you wrote with one
sentence each, state how a reader compiles and runs your program (also
put this in `code/README.md`), and describe how a run is configured
(command-line options, or which constants to edit and recompile).

*Implementation notes (B1-B7):* …

## Part C: Validation and analysis

*In the oral you should be able to:* say what the Horizons comparison
validates that the conservation tests cannot; predict how each error
changes when the time step is halved; explain why the Apophis prediction
is so much more demanding than the planetary orbits.

Every figure below needs axis labels with units, a legend where more
than one curve appears, and a caption or a sentence in the text stating
what it shows and what the reader should conclude from it. Report
numbers with a sensible number of digits: significant, not all of them.

### C1) Sanity checks

Simulate Sun and Earth (2 bodies), then Sun, Earth and Moon (3 bodies),
using velocity-Verlet.

Deliver:

- a **figure** of the Earth's orbit around the Sun in the ecliptic plane
  ($x$ versus $y$ in AU, equal aspect ratio, Sun marked);
- a **figure** of the Earth-Moon distance against time for the 3-body
  run, over at least three months;
- a **table** with your fitted orbital periods next to the literature
  values (sidereal year 365.256 d, sidereal month 27.322 d) and the
  deviation of each;
- a **figure** of the relative energy error against time for both runs.

Then comment: before trusting the second decimal of the fitted year,
think about what the snapshot's Earth velocity contains. It is the state
of the Earth *body*, in a system from which you have removed bodies.
Does that raise or lower the period, and by roughly how much?

*Results (C1):* …

In [ ]:
# C1: orbits, Earth-Moon distance, period fits, energy error (load from data/)

### C2) One-year comparison against Horizons

Integrate the full 2026 snapshot for exactly one year and compare your
final positions, body by body, with `bodies_2027-09-01.dat`.

Deliver:

- a **figure** showing the state of the system: the positions of all
  bodies in the ecliptic plane at the start and after one year (two
  panels, or one panel with two marker styles). Use a scale on which the
  inner planets are visible, and say which bodies fall outside it;
- a **table** of $\lVert \mathbf{r}_\mathrm{sim} -
  \mathbf{r}_\mathrm{Horizons} \rVert$ per body for **at least three
  time steps** spanning a factor 10 or more;
- a **figure** of those discrepancies (bar chart, logarithmic vertical
  axis, one group of bars per body).

Then discuss: relative to what scale is an error small, for each body?
Which bodies improve when you halve $\Delta t$ and which do not, and
what does that tell you about where the remaining error comes from?

*A question to take to the oral:* the snapshots give the outer planets
as system barycentres with system masses (see the data cell). Estimate
what the one-year discrepancy of Jupiter, and of Pluto whose moon Charon
carries 11% of the system mass, would have been with planet-*centre*
states instead, and why no time step could have fixed it.

*Results (C2):* …

In [ ]:
# C2: system snapshot at t=0 and t=1yr; per-body discrepancy table and bar chart

### C3) Conservation tests

Run the full system for one year with **both** integrators at **at least
four** time steps each.

Deliver:

- a **table**: for every integrator and time step, the largest relative
  drift over the run in $E$, in $\lVert\mathbf{L}\rVert$, and in the
  centre-of-mass velocity;
- a **figure**, log-log, of the energy error against $\Delta t$ for both
  integrators, with the fitted slopes stated in the legend or the text;
- a **figure** of the relative energy error against *time* at one common
  time step for both integrators, chosen to show the qualitative
  difference between them.

State the observed orders and whether they match what you predicted in
A3 and A4. Explain the behaviour of $\lVert\mathbf{L}\rVert$ and of the
centre of mass: which of them is a property of the integrator, and which
of the way you wrote the force loop?

*Results (C3):* observed orders …

In [ ]:
# C3: drift table; energy error vs dt (log-log) with fitted slopes; error vs time

### C4) Long-time stability

Simulate Sun, Jupiter, Saturn, Uranus, Neptune and Pluto for 200,000
days (about 548 years) with velocity-Verlet.

Deliver:

- a **figure** of the orbits in the ecliptic plane over the whole run;
- a **figure** of the invariants against time (energy, and
  $\lVert\mathbf{L}\rVert$ or the centre-of-mass motion);
- a **table** of the smallest and largest distance to the Sun reached by
  each planet during the run, next to its literature perihelion and
  aphelion.

Then judge the result: do the orbits stay closed and bounded, or is
there a systematic trend? Does the energy error grow with time, and is
that consistent with what you found in C3? Explain how a symplectic
integrator can accumulate a phase error along the orbit while the energy
stays bounded.

*Results (C4):* …

In [ ]:
# C4: outer solar system orbits; invariants vs time; perihelion/aphelion table

### C5) The Apophis 2029 close approach

Integrate the full system from the 2026 epoch through April 2029 with
velocity-Verlet, and determine the **date and the miss distance**
(Earth centre to Apophis) of the closest approach.

Deliver:

- a **table**: for each time step you ran, the miss distance and the
  date and time of closest approach; the value you obtain in the limit
  $\Delta t \to 0$ (extrapolate, and say how); and the JPL Horizons
  value with its source;
- a **figure** of the Earth-Apophis distance against time around the
  encounter, with one curve per time step, and the radius of the
  geostationary ring (42,164 km) drawn as a reference line;
- a **number** from one control run: the miss distance you get when the
  Moon is left out of the simulation. Explain the size of the difference;
  it is larger than the Moon's own pull on Apophis, so what dominates?

Then discuss what limits the accuracy of your prediction: integration
error, or model error (bodies you do not include, effects you do not
model)? Support your answer with evidence from your own runs rather than
assertion, and state which effect you would add first if you had to
improve the prediction.

*Practical hints:* the encounter is fast. Make sure your output cadence
around April 2029 is fine enough to resolve the minimum, or better,
detect the minimum in code. Think about which bodies must be in the
simulation for the approach to come out right.

*Results (C5):* closest approach on … at … km (Horizons: …); dt study …

In [ ]:
# C5: distance vs time around the encounter for several dt; convergence table

## Lessons learned

What would you do differently if you started over? What surprised you?
What in your results still bothers you?

*Lessons learned:* …

## Hand-in

One zip per group, layout exactly:

```
<group>_A1.zip
├── A1.ipynb     this notebook, filled in and executed
│                (all outputs present; task cells untouched)
├── A1.html      python3 nb2html.py A1.ipynb
├── code/        all .c/.h (incl. vec3d.h) + README.md with build/run
│                instructions; no binaries
└── data/        all data files this notebook loads: your outputs AND
                 the provided snapshots, so the notebook runs as handed in
```

Long runs are done in the terminal; record the exact commands in the
*Build & run log* and let the notebook load the saved output from
`data/`. Handed in executed, the notebook must open and display without
rerunning anything.

**Before you submit,** check your own zip:

```
python3 check_my_submission.py <group>_A1.zip --release A1.ipynb
```

where `--release` is a freshly downloaded, unmodified `A1.ipynb`. It
verifies the layout, that the task cells are untouched, that every cell
has been executed, that your C code compiles, and that every path and
link in the notebook is relative and present in the zip. Fix what it
reports, then submit.